# Geocoding — Parcel Address Enrichment
**Ontario Distribution-Connected Solar Siting | 10+ MW Projects**

> **Standalone utility notebook** — runs independently from the phase pipeline.
> Can be executed any time after Phase 3 has completed for a county.

Reverse-geocode the centroid of each developable parcel using the Mapbox
Geocoding API (v6) to obtain structured address data. Assign a unique parcel
identifier. **Updates the existing `developable_parcels_{slug}` table in-place**
by adding new columns — no separate output table is created.

**Input / Output (same table)**
- `analysis.developable_parcels_{county_slug}` (Phase 3 output)

**Columns Added**
| Column | Type | Source |
|---|---|---|
| `solar_parcel_uid` | TEXT | UUID-5 from parcel identity fields |
| `lat` / `lon` | FLOAT | WGS84 centroid via `ST_PointOnSurface` |
| `main_address` | TEXT | Mapbox primary reverse-geocoded address |
| `secondary_addresses` | TEXT (JSON) | Nearby addresses array |
| `county` / `township` / `locality` / `province` | TEXT | Mapbox context hierarchy |

---
Change only `COUNTY_NAME` in the Configuration cell to reproduce for any county.

## 0 · Configuration & Dependencies

In [30]:
# ─────────────────────────────────────────────────────────────────────────────
# PROJECT PARAMETERS — edit this cell only to run for a different county
# ─────────────────────────────────────────────────────────────────────────────

import os

COUNTY_NAME = "Ottawa"          # Must match adm_district_township.name_2

# PostGIS connection
PG_CONN = os.environ["POSTGRES_CONNECTION_STRING"]

# Output schema — tables are named per step and county automatically
OUTPUT_SCHEMA = "analysis"

# CRS
CRS_WGS84   = 4326
CRS_NAD83   = 4269   # NAD83 geographic — bridge CRS
CRS_ONTARIO = 5321   # NAD83(CSRS) / Ontario MNR Lambert — all calculations

# Geocoding parameters
GEOCODE_BATCH_DELAY = 0.05      # seconds between API calls (≤ 600 req/min on free tier)
GEOCODE_ENDPOINT    = "https://api.mapbox.com/search/geocode/v6/reverse"

## 1 · Environment Setup & Utilities

In [31]:
import warnings
warnings.filterwarnings("ignore")

import uuid
import time
import json

import geopandas as gpd
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

engine = create_engine(PG_CONN)

# Load Mapbox token from environment
load_dotenv()
MAPBOX_TOKEN = os.environ["MAPBOX_TOKEN"]


def read_postgis(sql: str, geom_col: str = "geom") -> gpd.GeoDataFrame:
    """Execute a PostGIS query and return a WGS84 GeoDataFrame."""
    gdf = gpd.read_postgis(sql, engine, geom_col=geom_col)
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=CRS_WGS84)
    return gdf.to_crs(epsg=CRS_WGS84)


def save_to_postgis(gdf: gpd.GeoDataFrame, table: str, label: str) -> None:
    """Write a GeoDataFrame to PostGIS in EPSG:5321 and create a GiST spatial index."""
    geom_col = gdf.geometry.name
    gdf.to_crs(epsg=CRS_ONTARIO).to_postgis(
        name=table,
        con=engine,
        schema=OUTPUT_SCHEMA,
        if_exists="replace",
        index=False,
        chunksize=500
    )
    with engine.connect() as conn:
        conn.execute(text(
            f"CREATE INDEX IF NOT EXISTS {table}_geom_idx "
            f"ON {OUTPUT_SCHEMA}.{table} USING GIST({geom_col})"
        ))
        conn.commit()
    print(f"  {label:35s} → {OUTPUT_SCHEMA}.{table} "
          f"({len(gdf):,} rows, EPSG:{CRS_ONTARIO}, GiST index)")


county_slug = COUNTY_NAME.lower().replace(" ", "_")

with engine.connect() as conn:
    print("PostGIS:", conn.execute(text("SELECT PostGIS_Full_Version()")).scalar())
print(f"County : {COUNTY_NAME} (slug: {county_slug})")
print(f"Mapbox token loaded: {'Yes' if MAPBOX_TOKEN else 'No'}")

PostGIS: POSTGIS="3.6.1 0" [EXTENSION] PGSQL="170" GEOS="3.14.1-CAPI-1.20.5" SFCGAL="SFCGAL 2.2.0, CGAL 6.1, BOOST 1.79.0" PROJ="9.7.1 NETWORK_ENABLED=OFF URL_ENDPOINT=https://cdn.proj.org USER_WRITABLE_DIRECTORY=/tmp/proj DATABASE_PATH=/usr/local/pgsql/share/proj/proj.db" (compiled against PROJ 9.7.1) GDAL="GDAL 3.12.1 "Chicoutimi", released 2025/12/12" LIBXML="2.11.5" LIBJSON="0.17" LIBPROTOBUF="1.5.0" WAGYU="0.5.0 (Internal)" TOPOLOGY RASTER
County : Ottawa (slug: ottawa)
Mapbox token loaded: Yes


---
## Step 1 — Load Developable Parcels & Compute Centroids

Read the Phase 3 output and compute a WGS84 centroid per parcel. Uses
`ST_PointOnSurface` (guaranteed inside the polygon) rather than `ST_Centroid`
(which can fall outside concave shapes).

In [32]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

# Join developable_parcels (clean parcel outlines) with developable_area_parcels
# (lot-level attributes) to enrich the table for UID generation and geocoding
gdf_parcels = read_postgis(f"""
SELECT
    dp.ogf_id,
    dp.parcel_acre,
    dap.lot_ident,
    dap.concession_ident,
    dap.geographic_township_name,
    dap.land_use_designation,
    dap.station_name,
    dap.ldc_name,
    ST_Y(ST_Transform(ST_PointOnSurface(dp.geom), {CRS_WGS84})) AS lat,
    ST_X(ST_Transform(ST_PointOnSurface(dp.geom), {CRS_WGS84})) AS lon,
    ST_Transform(dp.geom, {CRS_WGS84}) AS geom
FROM {OUTPUT_SCHEMA}.developable_parcels_{county_slug} dp
LEFT JOIN (
    SELECT DISTINCT ON (parcel_ogf_id)
        parcel_ogf_id, lot_ident, concession_ident,
        geographic_township_name, land_use_designation,
        station_name, ldc_name
    FROM {OUTPUT_SCHEMA}.developable_area_parcels_{county_slug}
    ORDER BY parcel_ogf_id
) dap ON dp.ogf_id = dap.parcel_ogf_id
""")

print(f"Step 1 — Loaded developable parcels for '{COUNTY_NAME}':")
print(f"  Row count: {len(gdf_parcels):,}")
print(f"  Columns  : {list(gdf_parcels.columns)}")
gdf_parcels.head(10)

Step 1 — Loaded developable parcels for 'Ottawa':
  Row count: 57
  Columns  : ['ogf_id', 'parcel_acre', 'lot_ident', 'concession_ident', 'geographic_township_name', 'land_use_designation', 'station_name', 'ldc_name', 'lat', 'lon', 'geom']


,ogf_id,parcel_acre,lot_ident,concession_ident,geographic_township_name,land_use_designation,station_name,ldc_name,lat,lon,geom
0,1900587127,94.016049,LOT 14,CON 1,CUMBERLAND,Rural Countryside,WILHAVEN DS,Hydro One Networks Inc.,45.434371,-75.297647,"MULTIPOLYGON (((-75.29505 45.43787, -75.30264 ..."
1,1900648194,111.307931,LOT 17,CON 2,OSGOODE,Rural Countryside,GREELY DS,Hydro One Networks Inc.,45.194165,-75.595464,"MULTIPOLYGON (((-75.59057 45.19847, -75.59063 ..."
2,1900706869,169.600353,LOT 22,CON 1,GOULBOURN,Rural Countryside,Richmond South MTS,Hydro Ottawa Limited,45.160383,-75.822476,"MULTIPOLYGON (((-75.82533 45.16664, -75.827 45..."
3,1900835830,94.552899,LOT 10,CON 8,GOULBOURN,Rural Countryside,Janet King DS 28kV,Hydro Ottawa Limited,45.185116,-75.971649,"MULTIPOLYGON (((-75.97754 45.19087, -75.97955 ..."
4,1900839277,153.456056,LOT 10,CON 7,GOULBOURN,Rural Countryside,Janet King DS 28kV,Hydro Ottawa Limited,45.175268,-75.960745,"MULTIPOLYGON (((-75.96458 45.18166, -75.96755 ..."
5,1900913845,65.300207,LOT 15,CON 2 FROM RIDEAU RIVER,GLOUCESTER,Greenbelt Rural,"Limebank MS, Uplands MS",Hydro Ottawa Limited,45.301331,-75.672729,"MULTIPOLYGON (((-75.67042 45.3049, -75.67652 4..."
6,1900931004,178.910278,LOT 10,CON 10,GOULBOURN,Rural Countryside,Janet King DS 28kV,Hydro Ottawa Limited,45.204682,-75.993712,"MULTIPOLYGON (((-75.99589 45.21061, -76.00156 ..."
7,1900932989,99.962063,LOT 9,CON 8,GOULBOURN,Rural Countryside,Janet King DS 28kV,Hydro Ottawa Limited,45.184193,-75.975625,"MULTIPOLYGON (((-75.98041 45.18909, -75.98252 ..."
8,1900934315,179.646470,LOT 11,CON 10,GOULBOURN,Rural Countryside,Janet King DS 28kV,Hydro Ottawa Limited,45.207724,-75.986987,"MULTIPOLYGON (((-75.9901 45.21428, -75.9957 45..."
9,1900934950,407.104933,LOT 9,CON 9,GOULBOURN,Rural Countryside,Janet King DS 28kV,Hydro Ottawa Limited,45.192342,-75.991633,"MULTIPOLYGON (((-75.98752 45.2, -75.98827 45.2..."


---
## Step 2 — Generate Deterministic Parcel UIDs

Create a deterministic **unique parcel identifier** (`solar_parcel_uid`) by
hashing the county slug, lot identity, concession, and parcel ID using UUID-5.
This is distinct from the cadastre `parcel_id` / `parcel_ogf_id` and is stable
across re-runs — the same input always produces the same UUID.

In [33]:
def make_parcel_uid(row):
    """Deterministic UUID-5 from parcel identity fields."""
    key = f"{county_slug}|{row['lot_ident']}|{row['concession_ident']}|{row['ogf_id']}"
    return str(uuid.uuid5(uuid.NAMESPACE_URL, key))

gdf_parcels["solar_parcel_uid"] = gdf_parcels.apply(make_parcel_uid, axis=1)

print(f"Step 2 — Generated {len(gdf_parcels):,} parcel UIDs")
print(f"  Unique UIDs: {gdf_parcels['solar_parcel_uid'].nunique():,}")
print(f"\nSample UIDs:")
gdf_parcels[["lot_ident", "concession_ident", "ogf_id", "solar_parcel_uid"]].head(5)

Step 2 — Generated 57 parcel UIDs
  Unique UIDs: 57

Sample UIDs:


,lot_ident,concession_ident,ogf_id,solar_parcel_uid
0,LOT 14,CON 1,1900587127,86ba30d1-d6e2-5cb8-bd7c-01603c565eb2
1,LOT 17,CON 2,1900648194,e6172b49-06fd-59ad-8ef5-d7eccae162c9
2,LOT 22,CON 1,1900706869,0cbc3168-35ee-5604-9ae9-905fd5f8aaa7
3,LOT 10,CON 8,1900835830,ec5f7fdb-6f7c-58d7-a765-04eaec8c651d
4,LOT 10,CON 7,1900839277,237b078f-7928-5f23-bee9-b1ee255deee2


---
## Step 3 — Reverse-Geocode via Mapbox API

Call the Mapbox v6 reverse endpoint for each parcel centroid. Parse the
structured response to extract address components.

- **Primary address**: `full_address` from the first feature
- **Secondary addresses**: `full_address` from remaining features (JSON array)
- **Context fields**: `district.name` → county, `place.name` → township,
  `locality.name` → locality, `region.name` → province

Individual failures are caught and logged — the loop does not abort.
Progress is printed every 50 rows.

In [ ]:
def reverse_geocode(lat, lon, token, types="address"):
    """Call Mapbox v6 reverse geocode. Returns parsed dict."""
    resp = requests.get(
        GEOCODE_ENDPOINT,
        params={
            "longitude": lon,
            "latitude": lat,
            "access_token": token,
            "types": types,
            "limit": 5,
        },
        timeout=10,
    )
    resp.raise_for_status()
    return resp.json()


def parse_features(data):
    """Extract address components from Mapbox v6 response features."""
    features = data.get("features", [])
    if not features:
        return None, None, {}
    primary   = features[0]["properties"]
    context   = primary.get("context", {})
    main_addr = primary.get("full_address", None)
    secondary = [
        f["properties"].get("full_address")
        for f in features[1:]
        if f["properties"].get("full_address")
    ]
    return main_addr, secondary, context


# Mapbox fallback strategy: try "address" first, then broaden to coarser types
# Rural parcels (farmland, forests) often have no civic address — fall back to
# place/locality/neighborhood which will still give township + county context
TYPES_PRIMARY  = "address"
TYPES_FALLBACK = "place,locality,neighborhood,address"

records = []
total = len(gdf_parcels)
fallback_count = 0

for idx, row in gdf_parcels.iterrows():
    try:
        # First attempt: address only
        data = reverse_geocode(row["lat"], row["lon"], MAPBOX_TOKEN, TYPES_PRIMARY)
        main_addr, secondary, context = parse_features(data)

        # Fallback: if no address found, widen types to include place/locality
        if main_addr is None:
            data = reverse_geocode(row["lat"], row["lon"], MAPBOX_TOKEN, TYPES_FALLBACK)
            main_addr, secondary, context = parse_features(data)
            if main_addr is not None:
                fallback_count += 1
            time.sleep(GEOCODE_BATCH_DELAY)  # extra delay for the second call

        records.append({
            "_idx":                idx,
            "main_address":        main_addr,
            "secondary_addresses": json.dumps(secondary) if secondary else None,
            "county":              context.get("district", {}).get("name"),
            "township":            context.get("place", {}).get("name"),
            "locality":            context.get("locality", {}).get("name"),
            "province":            context.get("region", {}).get("name"),
        })
    except Exception as e:
        print(f"  [{idx+1}/{total}] FAILED: {e}")
        records.append({
            "_idx":                idx,
            "main_address":        None,
            "secondary_addresses": None,
            "county":              None,
            "township":            None,
            "locality":            None,
            "province":            None,
        })

    if (idx + 1) % 50 == 0:
        print(f"  [{idx+1}/{total}] geocoded")

    time.sleep(GEOCODE_BATCH_DELAY)

print(f"\nGeocoding complete: {len(records)} parcels processed")
print(f"  Direct address hits : {sum(1 for r in records if r['main_address'] and r['_idx'] not in [])}")
print(f"  Fallback resolved   : {fallback_count}")
print(f"  Still missing       : {sum(1 for r in records if r['main_address'] is None)}")

---
## Step 4 — Merge Geocoding Results into GeoDataFrame

Assign geocoding columns back into `gdf_parcels` from the records list.

In [35]:
df_geo = pd.DataFrame(records).set_index("_idx")
for col in ["main_address", "secondary_addresses", "county", "township", "locality", "province"]:
    gdf_parcels[col] = df_geo[col]

geocoded_count = gdf_parcels["main_address"].notna().sum()
missing_count  = gdf_parcels["main_address"].isna().sum()

print(f"Step 4 — Merged geocoding results:")
print(f"  With main address: {geocoded_count:,}")
print(f"  Missing address  : {missing_count:,}")

gdf_parcels[["lot_ident", "ogf_id", "lat", "lon", "main_address", "county", "township"]].head(10)

Step 4 — Merged geocoding results:
  With main address: 56
  Missing address  : 1


,lot_ident,ogf_id,lat,lon,main_address,county,township
0,LOT 14,1900587127,45.434371,-75.297647,"4049 Birchgrove Road, Sarsfield, Ontario K0A 3...",None,Sarsfield
1,LOT 17,1900648194,45.194165,-75.595464,"2408 Manotick Station Road, Osgoode, Ontario K...",None,Osgoode
2,LOT 22,1900706869,45.160383,-75.822476,"6417 Bowrin Road, Richmond, Ontario K0A 2Z0, C...",None,Richmond
3,LOT 10,1900835830,45.185116,-75.971649,"2200 Munster Road, Ashton, Ontario K0A 1B0, Ca...",None,Ashton
4,LOT 10,1900839277,45.175268,-75.960745,"7840 Fallowfield Road, Ashton, Ontario K0A 1B0...",None,Ashton
5,LOT 15,1900913845,45.301331,-75.672729,"3995 Limebank Road, Gloucester, Ontario K1V 1E...",None,Gloucester
6,LOT 10,1900931004,45.204682,-75.993712,"7945 Fernbank Road, Ashton, Ontario K0A 1B0, C...",None,Ashton
7,LOT 9,1900932989,45.184193,-75.975625,"7910 Flewellyn Road, Ashton, Ontario K0A 1B0, ...",None,Ashton
8,LOT 11,1900934315,45.207724,-75.986987,"7731 Fernbank Road, Stittsville, Ontario K2S 1...",None,Stittsville
9,LOT 9,1900934950,45.192342,-75.991633,"7913 Flewellyn Road, Ashton, Ontario K0A 1B0, ...",None,Ashton


---
## Step 5 — Save Updated Table to PostGIS

Overwrite `developable_parcels_{slug}` with all original columns plus the new
geocoding columns. `save_to_postgis()` replaces the table atomically.

All original Phase 3 columns are preserved; the new columns
(`solar_parcel_uid`, `lat`, `lon`, `main_address`, `secondary_addresses`,
`county`, `township`, `locality`, `province`) are appended.

In [36]:
save_to_postgis(gdf_parcels, f"developable_parcels_{county_slug}",
                "Geocoding — Parcels + addresses")

  Geocoding — Parcels + addresses     → analysis.developable_parcels_ottawa (57 rows, EPSG:5321, GiST index)


---
## Step 6 — Verification & Summary

Confirm geocoding results and verify all expected columns exist in the DB table.

In [38]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")
print(f"Geocoded parcels: {len(gdf_parcels)}")
print(f"With main address:    {gdf_parcels['main_address'].notna().sum()}")
print(f"Missing address:      {gdf_parcels['main_address'].isna().sum()}")
print(f"Unique counties:      {gdf_parcels['county'].dropna().unique().tolist()}")
print(f"Unique townships:     {gdf_parcels['township'].dropna().unique().tolist()}")

# Verify columns exist in DB
with engine.connect() as conn:
    cols = conn.execute(text(f"""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = '{OUTPUT_SCHEMA}'
          AND table_name   = 'developable_parcels_{county_slug}'
        ORDER BY ordinal_position
    """)).fetchall()
    print(f"\nColumns in DB table ({OUTPUT_SCHEMA}.developable_parcels_{county_slug}):")
    for c in cols:
        print(f"  {c[0]}")

# Show parcels with missing addresses for retry identification
missing = gdf_parcels[gdf_parcels["main_address"].isna()]
if len(missing) > 0:
    print(f"\nParcels with missing addresses ({len(missing)}):")
    print(missing[["lot_ident", "concession_ident", "ogf_id", "lat", "lon"]].to_string())
else:
    print("\nAll parcels geocoded successfully — no missing addresses.")

Geocoded parcels: 57
With main address:    56
Missing address:      1
Unique counties:      []
Unique townships:     ['Sarsfield', 'Osgoode', 'Richmond', 'Ashton', 'Gloucester', 'Stittsville', 'Greely', 'Cumberland', 'Metcalfe', 'Dunrobin', 'Kanata', 'Smiths Falls', 'Nepean', 'Vars', 'Navan', 'Ottawa']

Columns in DB table (analysis.developable_parcels_ottawa):
  ogf_id
  parcel_acre
  lot_ident
  concession_ident
  geographic_township_name
  land_use_designation
  station_name
  ldc_name
  lat
  lon
  geom
  solar_parcel_uid
  main_address
  secondary_addresses
  county
  township
  locality
  province

Parcels with missing addresses (1):
   lot_ident concession_ident      ogf_id       lat       lon
33    LOT 21            CON 3  1921281296  45.29553 -75.77568
